# Combine and compare result files [^download]

[^download]: This notebook can be downloaded as **{nb-download}
repr(example_LF.ipynb) **.

This notebook shows how to load several APLOSE result files as separate `DetectionData` objects and compare their detections with time-series, heatmap, and scatter plots. The examples use three bundled whale-annotation datasets; adapt the file paths and label selections for your own data.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from pandas import Timedelta
from pandas.tseries import frequencies

from disclose.dataclass.detection_data import DetectionData

## Load data

Each dictionary describes one result file. Passing `merge=False` keeps the three datasets separate so they can be plotted side by side. The coordinates and timezone are set on each object for the seasonal and day/night overlays.

In [ ]:
config_list = [
    {
        "detection_file": Path(r"_static/ELAN_results.csv"),
        "timebin_new": Timedelta("1h"),
        "annotator": "lleboul",
        "filename_format": "%Y_%m_%d_%H_%M_%S_%f%z",
        "recording_file": Path(r"_static\ELAN_recording_periods.csv"),
    },
    {
        "detection_file": Path(r"_static/NEAMS_results.csv"),
        "timebin_new": Timedelta("1h"),
        "annotator": "lleboul",
        "filename_format": "%Y_%m_%d_%H_%M_%S_%f%z",
    },
    {
        "detection_file": Path(r"_static/MADW_results.csv"),
        "timebin_new": Timedelta("1h"),
        "annotator": "lleboul",
        "filename_format": "%Y_%m_%d_%H_%M_%S_%f%z",
    },
]

data_list = DetectionData.from_dict(config_list, merge=False)

for data in data_list:
    data.lat = -49
    data.lon = 70
    data.change_tz("Indian/Kerguelen")

## Overview

Inspect the labels and annotator in the first dataset before selecting a label for the plots below.

In [ ]:
print(data_list[0].overview(annotator="lleboul"))

## Weekly detections in one dataset

Display weekly detections for one label, with recording effort and season overlays.

In [ ]:
data = data_list[0]
fig, ax = plt.subplots(figsize=(10, 4))

data.plot.histogram(
    ax=ax,
    annotator="lleboul",
    label="Madagascan pygmy blue whale chorus",
    bin_size=frequencies.to_offset("1W-SUN"),
)
data.plot.set_ax(
    ax,
    date_format="%b %Y",
)
data.plot.feature(
    ax,
    season=True,
    legend=False,
)
plt.show()

## Compare datasets

Use the same label, bin width, and colors to compare weekly detections across all three result files.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 8))
colors = ["red", "green", "blue"]

for i in range(len(axs)):
    data = data_list[i]
    data.plot.histogram(
        ax=axs[i],
        annotator="lleboul",
        label="Antarctic blue whale chorus",
        bin_size=frequencies.to_offset("1W-SUN"),
        color=colors[i],
    )
    data.plot.set_ax(
        ax=axs[i],
        x_ticks_res=frequencies.to_offset("1MS"),
        date_format="%b %y",
    )
    data.plot.feature(axs[i], season=True, legend=False)

plt.show()

## Heatmaps by dataset

Each heatmap shows the selected label by hour of day and date. The daily cycle can help reveal changes in activity timing across datasets.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(8, 10))

for i in range(len(axs)):
    data_list[i].plot.heatmap(
        ax=axs[i],
        annotator="lleboul",
        label="Madagascan pygmy blue whale chorus",
        bin_size=frequencies.to_offset("1W-SUN"),
    )
    data_list[i].plot.feature(axs[i], rise_set=True, legend=False)
plt.show()

## Scatter plot

Plot monthly detections for two labels in each dataset. The time-axis tick interval is independent of the aggregation bin size.

In [ ]:
fig, axs = plt.subplots(3, 1, figsize=(10, 8))

for i in range(len(axs)):
    data = data_list[i]
    data.plot.scatter(
        ax=axs[i],
        annotator="lleboul",
        label=data.label[4:6],
        bin_size=frequencies.to_offset("1MS"),
        marker="x",
    )
    data.plot.set_ax(
        ax=axs[i],
        date_format="%b %Y",
        x_ticks_res=frequencies.to_offset("3MS"),
    )
    data.plot.feature(
        axs[i],
        rise_set=True,
        season=True,
    )

plt.show()